# 01 — Data Gathering (Live NOAA NDBC) → S3 (Curated)

This notebook **downloads live NOAA NDBC buoy data** (no backup files) and writes:

- **Parquet** to S3 for Athena / analytics
- **CSV** to S3 for simple pipeline ingestion

It also writes a small **manifest** so downstream notebooks know what was produced.


In [1]:
# Install deps (run once per kernel)
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

# Make local package imports work
repo_root = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(repo_root))

import json
import time
import boto3
import sagemaker
import pandas as pd

from src.data_utils import fetch_ndbc_data, clean_ndbc_data
from src.s3_utils import get_s3_client, get_sagemaker_bucket, verify_bucket, upload_df_to_s3

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# AWS context
sess = sagemaker.Session()
region = boto3.Session().region_name
bucket = get_sagemaker_bucket(sess)
s3_client = get_s3_client(region_name=region)

print("Region:", region)
print("Default bucket:", bucket)

verify_bucket(bucket, s3_client=s3_client)

Region: us-east-1
Default bucket: sagemaker-us-east-1-115800714036
[INFO] Bucket 'sagemaker-us-east-1-115800714036' exists.


True

In [4]:
# ---- CONFIG (edit these for your project) ----
BUOY_IDS = [
    "46086",    "46042",    "46011"
]
#

START_DATE = "2023-01-01"
END_DATE   = "2026-02-01"

S3_PREFIX_PARQUET = "curated/ndbc_parquet"
S3_PREFIX_CSV     = "curated/ndbc_csv"

#CURATED_PREFIX = "curated/ndbc"  # output prefix in S3

# File names inside each buoy=XXXX partition folder
PARQUET_NAME = "stdmet.parquet"
CSV_NAME     = "stdmet.csv"
# ----------------------------------------------

In [5]:
# Download -> clean -> upload (per buoy)
results = {}
row_counts = {}

for buoy_id in BUOY_IDS:
    print("\n" + "="*80)
    print("Buoy:", buoy_id)

    try:
        raw_df = fetch_ndbc_data(buoy_id, START_DATE, END_DATE, mode="stdmet")
    except Exception as e:
        print(f"[WARN] Fetch failed for buoy {buoy_id}: {e}")
        continue

    if raw_df is None or len(raw_df) == 0:
        print(f"[WARN] No data returned for buoy {buoy_id}. Skipping.")
        continue

    clean_df = clean_ndbc_data(raw_df, buoy_id)

    print("Rows (raw):", len(raw_df))
    print("Rows (clean):", len(clean_df))
    row_counts[buoy_id] = int(len(clean_df))

    # Upload Parquet + CSV
    parquet_uri = upload_df_to_s3(
        clean_df, bucket, S3_PREFIX_PARQUET, buoy_id,
        file_name=PARQUET_NAME,
        file_format="parquet",
        s3_client=s3_client,
    )
    csv_uri = upload_df_to_s3(
        clean_df, bucket, S3_PREFIX_CSV, buoy_id,
        file_name=CSV_NAME,
        file_format="csv",
        s3_client=s3_client,
    )

    results[buoy_id] = {"parquet": parquet_uri, "csv": csv_uri}
    

print("\nDONE. Uploaded buoys:", list(results.keys()))
results


Buoy: 46086
Rows (raw): 161892
Rows (clean): 51531
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46086/stdmet.parquet
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46086/stdmet.csv

Buoy: 46042
Rows (raw): 47137
Rows (clean): 12571
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46042/stdmet.parquet
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46042/stdmet.csv

Buoy: 46011
Rows (raw): 162193
Rows (clean): 53530
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46011/stdmet.parquet
[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46011/stdmet.csv

DONE. Uploaded buoys: ['46086', '46042', '46011']


{'46086': {'parquet': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46086/stdmet.parquet',
  'csv': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46086/stdmet.csv'},
 '46042': {'parquet': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46042/stdmet.parquet',
  'csv': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46042/stdmet.csv'},
 '46011': {'parquet': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_parquet/buoy=46011/stdmet.parquet',
  'csv': 's3://sagemaker-us-east-1-115800714036/curated/ndbc_csv/buoy=46011/stdmet.csv'}}

In [6]:
# Quick dataset sanity checks (AAI-540 rule of thumb: 3-5 files, 2 files >= 10k rows)
print("Row counts per buoy:")
display(pd.Series(row_counts, name="rows").sort_values(ascending=False))

num_files = len(results)
num_10k = sum(1 for v in row_counts.values() if v >= 10_000)
print("\nFiles uploaded:", num_files)
print("Files with >=10k rows:", num_10k)

if num_files < 3:
    raise RuntimeError("Need at least 3 buoy files uploaded to meet dataset size guidance. Adjust BUOY_IDS.")
if num_10k < 2:
    raise RuntimeError("Need at least 2 buoy files with >=10k rows. Expand date range or choose different buoys.")

Row counts per buoy:


46011    53530
46086    51531
46042    12571
Name: rows, dtype: int64


Files uploaded: 3
Files with >=10k rows: 3


In [7]:
# Write a manifest (local + S3)
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "bucket": bucket,
    "region": region,
    "S3_PREFIX_PARQUET" : S3_PREFIX_PARQUET,
    "S3_PREFIX_CSV" : S3_PREFIX_CSV,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "buoy_ids": list(results.keys()),
    "row_counts": row_counts,
    "artifacts": results,
}

manifest_path = repo_root / "manifests"
manifest_path.mkdir(exist_ok=True)
local_manifest = manifest_path / "curated_manifest.json"
local_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Upload manifest too
manifest_s3_uri = upload_df_to_s3(
    pd.DataFrame([manifest]),
    bucket=bucket,
    s3_prefix="manifests",
    buoy_id="all",
    file_name="curated_manifest.csv",
    file_format="csv",
    s3_client=s3_client,
)

print("Local manifest:", local_manifest)
print("S3 manifest CSV:", manifest_s3_uri)

[INFO] Uploaded to s3://sagemaker-us-east-1-115800714036/manifests/buoy=all/curated_manifest.csv
Local manifest: /home/sagemaker-user/buoyCast-main/manifests/curated_manifest.json
S3 manifest CSV: s3://sagemaker-us-east-1-115800714036/manifests/buoy=all/curated_manifest.csv


In [8]:
# Store variables for downstream notebooks
%store bucket
%store region
%store S3_PREFIX_PARQUET
%store S3_PREFIX_CSV
%store BUOY_IDS
%store manifest_s3_uri

Stored 'bucket' (str)
Stored 'region' (str)
Stored 'S3_PREFIX_PARQUET' (str)
Stored 'S3_PREFIX_CSV' (str)
Stored 'BUOY_IDS' (list)
Stored 'manifest_s3_uri' (str)
